In [ ]:

# RouteRanker - AI-Powered Public Transit Route Optimization

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os

print("RouteRanker project started!")
print("Libraries imported successfully.")


RouteRanker project started!
Libraries imported successfully.


In [ ]:

# Check the GTFS files required for RouteRanker

required_files = [
    "agency.txt",
    "routes.txt",
    "trips.txt",
    "stops.txt",
    "stop_times.txt",
    "calendar.txt"
]

print("GTFS files we need to check:")
for file in required_files:
    print("-", file)


GTFS files we need to check:
- agency.txt
- routes.txt
- trips.txt
- stops.txt
- stop_times.txt
- calendar.txt


In [ ]:
import os
print(os.listdir("/content"))

['.config', 'sample_data']


In [ ]:
import os

for root, dirs, files in os.walk("/contens"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import requests

url = "https://raw.githubusercontent.com/croyla/pmpml-gtfs/refs/heads/main/pmpml_gtfs_compat.zip"

response = requests.get(url, timeout=120)

print("Status code:", response.status_code)
print("File size:", round(len(response.content) / (1024 * 1024), 2), "MB")

Status code: 200
File size: 4.63 MB


In [ ]:
import requests

url = "https://raw.githubusercontent.com/croyla/pmpml-gtfs/refs/heads/main/pmpml_gtfs_compat.zip"
zip_path = "/content/pmpml_gtfs_compat.zip"

response = requests.get(url, timeout=120)
response.raise_for_status()

with open(zip_path, "wb") as f:
    f.write(response.content)

print("Download completed!")
print("File size:", round(len(response.content) / (1024 * 1024), 2), "MB")

Download completed!
File size: 4.63 MB


In [ ]:
import zipfile
import os

zip_path = "/content/pmpml_gtfs_compat.zip"
extract_path = "/content/pune_gtfs"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print("\nFiles available:")

for file in os.listdir(extract_path):
    print(file)

Dataset extracted successfully!

Files available:
calendar.txt
stops.txt
feed_info.txt
shapes.txt
routes.txt
trips.txt
agency.txt
stop_times.txt


In [ ]:

import pandas as pd
import os

data_path = "/content/pune_gtfs"

stops = pd.read_csv(os.path.join(data_path, "stops.txt"))
routes = pd.read_csv(os.path.join(data_path, "routes.txt"))
trips = pd.read_csv(os.path.join(data_path, "trips.txt"))
stop_times = pd.read_csv(os.path.join(data_path, "stop_times.txt"))
calendar = pd.read_csv(os.path.join(data_path, "calendar.txt"))

print("PUNE GTFS DATASET SUMMARY")
print("-------------------------")
print("Total Stops:", len(stops))
print("Total Routes:", len(routes))
print("Total Trips:", len(trips))
print("Total Stop Times:", len(stop_times))
print("Calendar Records:", len(calendar))
print("\nDataset loaded successfully!")


PUNE GTFS DATASET SUMMARY
-------------------------
Total Stops: 6745
Total Routes: 315
Total Trips: 10128
Total Stop Times: 429302
Calendar Records: 1

Dataset loaded successfully!


In [ ]:

print("Calendar columns:")
print(calendar.columns.tolist())

print("\nCalendar data:")
display(calendar)

print("\nFeed information:")
feed_info = pd.read_csv("/content/pune_gtfs/feed_info.txt")
display(feed_info)

Calendar columns:
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']

Calendar data:


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,WEEKDAY,1,1,1,1,1,1,1,20261007,20270405



Feed information:


,feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_contact_url
0,BLRTransit,https://blrtransit.com,en,20261007,20270405,https://blrtransit.com


In [ ]:

print("AGENCY INFORMATION")
display(pd.read_csv("/content/pune_gtfs/agency.txt"))

print("\nSAMPLE STOPS")
display(stops[["stop_id", "stop_name"]].head(10))

print("\nSAMPLE ROUTES")
display(routes.head(5))


AGENCY INFORMATION


,agency_id,agency_name,agency_url,agency_timezone,agency_phone,agency_email
0,PMPML,Pune Mahanagar Parivahan Mahamandal Limited,https://www.pmpml.org,Asia/Kolkata,+91 02024545454,complaints@pmpml.org



SAMPLE STOPS


,stop_id,stop_name
0,106793,Pune Station Moledina Road
1,319,Income Tax Office
2,320,Gpo
3,321,Collector Kacheri
4,322,Zilla Parishad
5,323,15 August Chowk
6,324,Apollo Talkies
7,325,Phadke Haud
8,326,Jijamatabaug
9,327,Vasant Talkies



SAMPLE ROUTES


,route_id,agency_id,route_short_name,route_long_name,route_type
0,100,PMPML,100,Hinjawadi Maan Phase 3 ⇆ Ma Na Pa Dengle Pul N...,3
1,103,PMPML,103,Kothrud Depot ⇆ Katraj,3
2,104,PMPML,104,Dsk Vishwa ⇆ Shivaji Nagar,3
3,107,PMPML,107,Pimplegurav ⇆ Ganpati Matha,3
4,108,PMPML,108,Pune Station Moledina Road ⇆ Sutardara,3


In [ ]:

# 1. Each trip ki enni stops unnayo calculate cheddham
trip_stop_counts = stop_times.groupby("trip_id")["stop_id"].count()

# 2. Trip counts ni route-wise calculate cheddham
route_trip_counts = trips.groupby("route_id")["trip_id"].nunique()

# 3. Each route lo unique stops count
route_stop_counts = (
    trips[["route_id", "trip_id"]]
    .merge(stop_times[["trip_id", "stop_id"]], on="trip_id")
    .groupby("route_id")["stop_id"]
    .nunique()
)

# 4. Combine results
route_analysis = pd.DataFrame({
    "total_trips": route_trip_counts,
    "unique_stops": route_stop_counts
}).fillna(0).astype(int)

route_analysis = route_analysis.reset_index().merge(
    routes[["route_id", "route_short_name", "route_long_name"]],
    on="route_id",
    how="left"
)

print("ROUTE ANALYSIS")
display(route_analysis.head(10))

print("\nTotal routes analyzed:", len(route_analysis))


ROUTE ANALYSIS


,route_id,total_trips,unique_stops,route_short_name,route_long_name
0,100,79,95,100,Hinjawadi Maan Phase 3 ⇆ Ma Na Pa Dengle Pul N...
1,103,29,35,103,Kothrud Depot ⇆ Katraj
2,104,11,44,104,Dsk Vishwa ⇆ Shivaji Nagar
3,107,6,111,107,Pimplegurav ⇆ Ganpati Matha
4,108,6,56,108,Pune Station Moledina Road ⇆ Sutardara
5,11,40,82,11,Pimplegurav ⇆ Marketyard Depot
6,111M,4,74,111M,Ma Na Pa Congress Bhavan ⇆ Bhekrai Nagar
7,113,11,50,113,Sangvi Vasantdada Putala ⇆ Vasant Talkies
8,114,116,82,114,Bajirao Padale Chowk Mhalunge ⇆ Ma Na Pa Main ...
9,115,58,106,115,Hinjawadi Maan Phase 3 ⇆ Pune Station Moledina...



Total routes analyzed: 315


In [ ]:

# Routes with the fewest scheduled trips
lowest_trip_routes = route_analysis.sort_values(
    by="total_trips",
    ascending=True
)

print("10 ROUTES WITH FEWEST TRIPS")
display(
    lowest_trip_routes[
        ["route_short_name", "route_long_name",
         "total_trips", "unique_stops"]
    ].head(10)
)

print("\n10 ROUTES WITH MOST TRIPS")
display(
    route_analysis.sort_values(
        by="total_trips",
        ascending=False
    )[
        ["route_short_name", "route_long_name",
         "total_trips", "unique_stops"]
    ].head(10)
)


10 ROUTES WITH FEWEST TRIPS


,route_short_name,route_long_name,total_trips,unique_stops
30,135,Ranjangaon Sandas ⇆ Ma Na Pa Shramik Bhavan,3,165
44,147,Nhavi Sandas ⇆ Ma Na Pa Shramik Bhavan,3,131
85,193,Khutbav Gaon ⇆ Hadapsar Gadital Solapur Road,3,143
215,360A,Aundhgaon ⇆ Alandi,3,76
116,232,Gram Panchayat Belawade ⇆ Marketyard Depot,3,157
134,279,Katar Khadak ⇆ Marketyard Depot,3,167
144,296A,Vangniwadi ⇆ Balaji Nagar Corner Satara Road,3,158
119,233B,Bhadasgaon ⇆ Marketyard Depot,3,164
104,212,Morgaon St Stand ⇆ Hadapsar Gadital,3,188
131,264,Pabal Gaon ⇆ Bhosari Pmt Chowk,3,149



10 ROUTES WITH MOST TRIPS


,route_short_name,route_long_name,total_trips,unique_stops
246,5,Pune Station Moledina Stand ⇆ Swargate,175,27
8,114,Bajirao Padale Chowk Mhalunge ⇆ Ma Na Pa Main ...,116,82
292,H8,Fursungi ⇆ Hadapsar Gadital,110,28
201,350,Sant Tukaram Maharaj Krida Sankul Dehugaon ⇆ N...,109,50
66,167,Kesnand Phata Wagholi ⇆ Bhekrai Nagar,107,66
228,372,Megapolis Phase 3 ⇆ Nigdi Bhakti Shakti Stand,106,87
150,301,Shewalewadi Depot ⇆ Katraj,105,62
241,42,Nigdi Bhakti Shakti ⇆ Gujarwadi Pmt Bus Stand,105,105
293,H9,Manjari Budruk ⇆ Hadapsar Gadital Solapur Road,103,28
177,324,Hinjawadi Maan Phase 3 ⇆ Bhosari Terminal,102,79


In [ ]:

# Find stops served by multiple routes

stop_route_counts = (
    trips[["route_id", "trip_id"]]
    .merge(stop_times[["trip_id", "stop_id"]], on="trip_id")
    .drop_duplicates(["route_id", "stop_id"])
    .groupby("stop_id")["route_id"]
    .nunique()
)

print("Total stops served:", len(stop_route_counts))
print("Stops served by multiple routes:", (stop_route_counts > 1).sum())

print("\nTop 15 shared stops:")
shared_stops = stop_route_counts[stop_route_counts > 1].sort_values(ascending=False).head(15)
display(
    shared_stops.rename("number_of_routes")
    .reset_index()
    .merge(stops[["stop_id", "stop_name"]], on="stop_id", how="left")
    [["stop_name", "number_of_routes"]]
)


Total stops served: 6745
Stops served by multiple routes: 3662

Top 15 shared stops:


,stop_name,number_of_routes
0,Income Tax Office,67
1,Swargate,61
2,Gpo,58
3,Pune Station Moledina Stand,56
4,Collector Kacheri,53
5,Lakshminarayan,49
6,Sndt Mahila Vidhyapith,46
7,Shivaji Maharaj Putla,45
8,St Colony,44
9,Ambedkar Bhavan,44


In [ ]:

print("trips exists:", "trips" in globals())
print("stop_times exists:", "stop_times" in globals())
print("stops exists:", "stops" in globals())

if "trips" in globals():
    print("Trips columns:", trips.columns.tolist())
if "stop_times" in globals():
    print("Stop_times columns:", stop_times.columns.tolist())
if "stops" in globals():
    print("Stops columns:", stops.columns.tolist())


trips exists: True
stop_times exists: True
stops exists: True
Trips columns: ['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'shape_id']
Stop_times columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'timepoint']
Stops columns: ['stop_id', 'stop_name', 'stop_lat', 'stop_lon']


In [ ]:

# Route overlap analysis: compare unique stops across routes

route_stop_pairs = (
    trips[["route_id", "trip_id"]]
    .merge(
        stop_times[["trip_id", "stop_id"]],
        on="trip_id",
        how="inner"
    )
    [["route_id", "stop_id"]]
    .drop_duplicates()
)

route_stop_sets = (
    route_stop_pairs.groupby("route_id")["stop_id"]
    .apply(set)
    .to_dict()
)

route_ids = list(route_stop_sets.keys())
overlaps = []

for i in range(len(route_ids)):
    for j in range(i + 1, len(route_ids)):
        r1, r2 = route_ids[i], route_ids[j]
        common_stops = route_stop_sets[r1] & route_stop_sets[r2]

        if len(common_stops) >= 10:
            overlaps.append({
                "route_1": r1,
                "route_2": r2,
                "shared_stops": len(common_stops)
            })

overlap_df = pd.DataFrame(overlaps)

if not overlap_df.empty:
    overlap_df = overlap_df.sort_values(
        "shared_stops", ascending=False
    ).reset_index(drop=True)

    overlap_df = overlap_df.merge(
        routes[["route_id", "route_short_name", "route_long_name"]]
        .rename(columns={
            "route_id": "route_1",
            "route_short_name": "route_1_name",
            "route_long_name": "route_1_description"
        }),
        on="route_1",
        how="left"
    )

    overlap_df = overlap_df.merge(
        routes[["route_id", "route_short_name", "route_long_name"]]
        .rename(columns={
            "route_id": "route_2",
            "route_short_name": "route_2_name",
            "route_long_name": "route_2_description"
        }),
        on="route_2",
        how="left"
    )

    print("Route pairs sharing 10 or more stops:", len(overlap_df))
    display(overlap_df.head(15))
else:
    print("No route pairs share 10 or more unique stops.")


Route pairs sharing 10 or more stops: 4517


,route_1,route_2,shared_stops,route_1_name,route_1_description,route_2_name,route_2_description
0,231,233A,179,231,Shilim Gaon ⇆ Marketyard Depot,233A,Kolwan Gaon ⇆ Marketyard Depot
1,233,70,137,233,Paud Shashkiy Vishram Gruh ⇆ Marketyard Depot,70,Mulshi Gaon ⇆ Marketyard Depot
2,231,233B,135,231,Shilim Gaon ⇆ Marketyard Depot,233B,Bhadasgaon ⇆ Marketyard Depot
3,233B,70,132,233B,Bhadasgaon ⇆ Marketyard Depot,70,Mulshi Gaon ⇆ Marketyard Depot
4,233A,233B,132,233A,Kolwan Gaon ⇆ Marketyard Depot,233B,Bhadasgaon ⇆ Marketyard Depot
5,135,137,132,135,Ranjangaon Sandas ⇆ Ma Na Pa Shramik Bhavan,137,Pargaon Salu Malu ⇆ Ma Na Pa Shramik Bhavan
6,231,233,131,231,Shilim Gaon ⇆ Marketyard Depot,233,Paud Shashkiy Vishram Gruh ⇆ Marketyard Depot
7,233,233B,131,233,Paud Shashkiy Vishram Gruh ⇆ Marketyard Depot,233B,Bhadasgaon ⇆ Marketyard Depot
8,233,233A,130,233,Paud Shashkiy Vishram Gruh ⇆ Marketyard Depot,233A,Kolwan Gaon ⇆ Marketyard Depot
9,232,70,130,232,Gram Panchayat Belawade ⇆ Marketyard Depot,70,Mulshi Gaon ⇆ Marketyard Depot


In [ ]:

# Route coverage statistics

print("=== PUNE ROUTE COVERAGE ===")

print("Total routes:", routes["route_id"].nunique())
print("Total unique stops:", stops["stop_id"].nunique())

# Number of unique stops served by each route
route_coverage = (
    route_stop_pairs.groupby("route_id")["stop_id"]
    .nunique()
    .reset_index(name="unique_stops")
    .merge(
        routes[["route_id", "route_short_name", "route_long_name"]],
        on="route_id",
        how="left"
    )
)

print("\nRoutes serving the most unique stops:")
display(
    route_coverage.sort_values("unique_stops", ascending=False)
    [["route_short_name", "route_long_name", "unique_stops"]]
    .head(10)
)

print("\nRoutes serving the fewest unique stops:")
display(
    route_coverage.sort_values("unique_stops")
    [["route_short_name", "route_long_name", "unique_stops"]]
    .head(10)
)

print("\nAverage unique stops per route:",
      round(route_coverage["unique_stops"].mean(), 2))


=== PUNE ROUTE COVERAGE ===
Total routes: 315
Total unique stops: 6745

Routes serving the most unique stops:


,route_short_name,route_long_name,unique_stops
115,231,Shilim Gaon ⇆ Marketyard Depot,212
260,65A,Patas ⇆ Hadapsar Gadital Solapur Road,193
63,164,Nhavare ⇆ Ma Na Pa Shramik Bhavan,190
58,159B,Loni Dhamni ⇆ Ma Na Pa Shramik Bhavan,190
143,296,Velhe ⇆ Balaji Nagar Corner Satara Road,188
104,212,Morgaon St Stand ⇆ Hadapsar Gadital,188
266,70,Mulshi Gaon ⇆ Marketyard Depot,184
118,233A,Kolwan Gaon ⇆ Marketyard Depot,183
223,368,Lonavala Railway Station ⇆ Nigdi Pawale Chowk,177
114,228,Vadgaon Mawal ⇆ Katraj,173



Routes serving the fewest unique stops:


,route_short_name,route_long_name,unique_stops
110,221,I V Estate Wagholi ⇆ Wagholi Kesnand Phata,10
89,2,Shivaji Nagar ⇆ Swargate,18
288,AP2,Upper Depot ⇆ Katraj Kondhwa Road,21
239,4,Shivaji Nagar ⇆ Swargate,21
296,K12A,Katraj ⇆ Nandanvan City Narhe,21
106,214,Shingad Collage Ambegaon ⇆ Katraj,22
299,K18,Katraj ⇆ Waghjai Nagar Shivaji Chowk,22
163,308,Gharkul Vasahat Chikhali ⇆ Nigdi Pawale Chowk,23
307,N15,Rupee Nagar ⇆ Nigdi Pawale Chowk,23
295,K12,Katraj ⇆ Abhinav Collage Narhe,23



Average unique stops per route: 82.09


In [ ]:
# Check arrival and departure time columns
print("Columns in stop_times:")
print(stop_times.columns.tolist())

# Display sample arrival and departure times
stop_times[[
    "trip_id",
    "stop_id",
    "stop_sequence",
    "arrival_time",
    "departure_time"
]].head(10)

Columns in stop_times:
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'timepoint']


,trip_id,stop_id,stop_sequence,arrival_time,departure_time
0,trip_18_0,106793,1,08:30:00,08:30:00
1,trip_18_0,319,2,08:31:00,08:31:00
2,trip_18_0,320,3,08:33:09,08:33:09
3,trip_18_0,321,4,08:34:09,08:34:09
4,trip_18_0,322,5,08:35:09,08:35:09
5,trip_18_0,323,6,08:36:27,08:36:27
6,trip_18_0,324,7,08:37:29,08:37:29
7,trip_18_0,325,8,08:38:45,08:38:45
8,trip_18_0,326,9,08:40:05,08:40:05
9,trip_18_0,327,10,08:41:12,08:41:12


In [ ]:
# Check missing arrival and departure times
print("Missing arrival times:", stop_times["arrival_time"].isna().sum())
print("Missing departure times:", stop_times["departure_time"].isna().sum())

# Check whether arrival and departure times differ
different_times = (
    stop_times["arrival_time"] != stop_times["departure_time"]
).sum()

print("Rows with different arrival and departure times:", different_times)

# Show records where arrival and departure times differ
stop_times.loc[
    stop_times["arrival_time"] != stop_times["departure_time"],
    ["trip_id", "stop_id", "stop_sequence",
     "arrival_time", "departure_time"]
].head(10)

Missing arrival times: 0
Missing departure times: 0
Rows with different arrival and departure times: 0


,trip_id,stop_id,stop_sequence,arrival_time,departure_time


In [ ]:
# Convert arrival time to seconds
def time_to_seconds(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

# Get first and last stop time for each trip
trip_times = stop_times.sort_values("stop_sequence").groupby("trip_id").agg(
    start_time=("departure_time", "first"),
    end_time=("arrival_time", "last"),
    total_stops=("stop_id", "count")
).reset_index()

# Calculate scheduled trip duration in minutes
trip_times["duration_minutes"] = (
    trip_times["end_time"].apply(time_to_seconds)
    - trip_times["start_time"].apply(time_to_seconds)
) / 60

print("Total trips analyzed:", len(trip_times))
print("\nTrip duration summary:")
print(trip_times["duration_minutes"].describe())

trip_times.head(10)

Total trips analyzed: 10128

Trip duration summary:
count    10128.000000
mean        65.172699
std         36.245098
min          7.683333
25%         37.100000
50%         61.400000
75%         82.900000
max        242.466667
Name: duration_minutes, dtype: float64


,trip_id,start_time,end_time,total_stops,duration_minutes
0,trip_1000_0,05:40:00,07:07:42,59,87.700000
1,trip_1000_1,06:00:00,07:27:42,59,87.700000
2,trip_1000_2,06:40:00,08:07:42,59,87.700000
3,trip_1000_3,07:40:00,09:07:42,59,87.700000
4,trip_1000_4,08:50:00,10:17:42,59,87.700000
5,trip_1000_5,09:40:00,11:07:42,59,87.700000
6,trip_1000_6,11:10:00,12:37:42,59,87.700000
7,trip_1000_7,12:40:00,14:07:42,59,87.700000
8,trip_1001_0,06:10:00,07:38:04,59,88.066667
9,trip_1001_1,07:10:00,08:38:04,59,88.066667
